# Блокнот 17. YOLO: letterbox, готовая модель и обучение на своих данных

Курс «Машинное обучение с нуля», модуль 17. Разделы: 1 — урок 17.3, 2 — уроки 17.4–17.6, 3 — урок 17.8. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

В Colab включите видеокарту: «Среда выполнения → Сменить среду выполнения → T4 GPU». Без неё всё работает, но обучение в разделе 3 идёт в уменьшенном виде (3 эпохи на картинках 320 × 320 вместо 30 эпох на 640 × 640).

Блокнот скачивает картинку COCO (≈170 КБ), веса YOLO26n и YOLO11n (по ≈5 МБ) и учебный набор coco8 (8 картинок, ≈1 МБ).

In [ ]:
%pip install -q ultralytics

In [ ]:
import os
import time
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from matplotlib.patches import Rectangle
from torchvision.ops import batched_nms, box_iou
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"
DATA = os.environ.get("COURSE_DATA", "data")
os.makedirs(DATA, exist_ok=True)
print("устройство:", device, "| ultralytics", ultralytics.__version__)


def draw(ax, boxes, labels=None, color="lime"):
    for k, (x1, y1, x2, y2) in enumerate(boxes):
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color, lw=2))
        if labels is not None:
            ax.text(x1, y1 - 3, labels[k], color="white", fontsize=8, bbox=dict(fc=color, ec="none", pad=1))

## 1. Letterbox и обратный пересчёт рамок (урок 17.3)

YOLO работает с квадратом `size × size`. Кадр уменьшают **с сохранением пропорций** так, чтобы большая сторона стала `size`, и ставят по центру серого квадрата (цвет 114, как в Ultralytics).

**Задание 1.** Допишите `letterbox_image`: верните квадратную картинку, коэффициент масштаба `r` и отступы `(pad_left, pad_top)`. Новые размеры — `round(w * r)`, `round(h * r)`; отступы — `(size - new_w) // 2` и `(size - new_h) // 2`.

In [ ]:
def letterbox_image(img, size=640, color=114):
    h, w = img.shape[:2]
    r = ...  # ваш код: во сколько раз уменьшить картинку
    new_w, new_h = ..., ...
    pad_left, pad_top = ..., ...
    out = ...  # серый квадрат np.full((size, size, 3), color, dtype=img.dtype), в центр — cv2.resize(img, (new_w, new_h))
    return out, r, (pad_left, pad_top)


# кадр камеры X-wing 1280 × 720 с одной «целью»
frame = np.full((720, 1280, 3), (40, 60, 90), dtype=np.uint8)
target = (800, 200, 1000, 400)  # xyxy в пикселях кадра
cv2.rectangle(frame, target[:2], target[2:], (230, 120, 40), -1)

square, r, (pl, pt) = letterbox_image(frame, 640)
assert square.shape == (640, 640, 3)
assert abs(r - 0.5) < 1e-9 and (pl, pt) == (0, 140)
assert (square[:140] == 114).all() and (square[-140:] == 114).all()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].imshow(frame)
axes[0].set_title("кадр 1280 × 720")
axes[1].imshow(square)
axes[1].set_title("letterbox 640 × 640: полосы 140 px сверху и снизу")
for ax in axes:
    ax.axis("off")
plt.show()

Сеть выдаёт рамки **в координатах квадрата**. Чтобы нарисовать их на исходном кадре, надо сделать обратное: вычесть отступы и разделить на `r`.

**Задание 2.** Допишите `unletterbox(boxes, r, pad, shape)`: `boxes` — массив `(n, 4)` в формате xyxy на квадрате. Верните рамки в пикселях исходного кадра, обрезанные по его границам (`np.clip`; `shape = (h, w)`).

In [ ]:
def unletterbox(boxes, r, pad, shape):
    boxes = np.asarray(boxes, dtype=float).copy()
    h, w = shape
    ...  # ваш код
    return boxes


# цель на квадрате: (800·0,5 + 0, 200·0,5 + 140, …) = (400, 240, 500, 340)
on_square = np.array([[400, 240, 500, 340], [-10, 100, 50, 150]])
back = unletterbox(on_square, r, (pl, pt), frame.shape[:2])
print(back)
assert np.allclose(back[0], target)
assert back[1, 0] == 0 and back[1, 1] == 0  # рамка за краем обрезана

# Ultralytics делает то же самое функцией scale_boxes
from ultralytics.utils.ops import scale_boxes

theirs = scale_boxes((640, 640), torch.tensor(on_square, dtype=torch.float32), frame.shape[:2]).numpy()
print("scale_boxes из Ultralytics:", theirs[0])
assert np.allclose(theirs[0], back[0], atol=1)

## 2. Готовая YOLO на картинке (уроки 17.4–17.6)

Возьмём ту же картинку COCO, что и в блокноте 16 (две кошки на диване), и YOLO26n — самую маленькую модель Ultralytics. Веса скачаются сами при первом вызове.

In [ ]:
img_path = os.path.join(DATA, "coco_000000039769.jpg")
if not os.path.exists(img_path):
    urllib.request.urlretrieve("http://images.cocodataset.org/val2017/000000039769.jpg", img_path)

model = YOLO("yolo26n.pt")
res = model.predict(img_path, conf=0.25, verbose=False)[0]
print("время, мс:", {k: round(v, 1) for k, v in res.speed.items()})
names = res.names
found = [names[int(c)] for c in res.boxes.cls]
for name, c, b in zip(found, res.boxes.conf.tolist(), res.boxes.xyxy.round().int().tolist()):
    print(f"{name:10s} {c:.2f} {b}")
assert "cat" in found

plt.figure(figsize=(7, 5))
plt.imshow(res.plot()[..., ::-1])  # plot() рисует в BGR, как OpenCV (урок 13.1) — переворачиваем каналы
plt.axis("off")
plt.title("YOLO26n: без NMS, одна рамка на объект")
plt.show()

n_params = sum(p.numel() for p in model.model.parameters())
print(f"параметров YOLO26n: {n_params / 1e6:.2f} млн")

Заглянем в **сырой выход** сети. Возьмём YOLO11n — у неё обычная голова с NMS. Подадим квадрат 640 × 640 из своего `letterbox_image`: в выходе 84 строки (4 числа рамки `cx, cy, w, h` в пикселях квадрата и 80 вероятностей классов) на **8400** точек — 80² + 40² + 20² (конспект модуля 17).

**Задание 3.** Допишите `decode(raw, conf, iou)`: из `raw` формы `(84, 8400)` возьмите для каждой точки лучший класс и его вероятность, оставьте точки с вероятностью больше `conf`, переведите `cx, cy, w, h` в xyxy и уберите дубли NMS по классам — `batched_nms(boxes, scores, classes, iou)` из torchvision (урок 16.2). Верните `boxes, scores, classes` оставшихся рамок.

In [ ]:
def decode(raw, conf=0.25, iou=0.7):
    ...  # ваш код


m11 = YOLO("yolo11n.pt")
net = m11.model.eval()
img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
sq, r2, pad2 = letterbox_image(img_rgb, 640)
x = torch.from_numpy(sq).permute(2, 0, 1)[None].float() / 255  # (1, 3, 640, 640), значения 0…1
with torch.no_grad():
    raw = net(x)[0][0]
print("форма сырого выхода:", tuple(raw.shape))
assert raw.shape == (84, 8400)

boxes, scores, classes = decode(raw)
mine = unletterbox(boxes.numpy(), r2, pad2, img_rgb.shape[:2])
print("наши рамки:", [f"{names[int(c)]} {s:.2f}" for c, s in zip(classes, scores)])

ref = m11.predict(img_path, conf=0.25, verbose=False)[0]
print("Ultralytics:", [f"{names[int(c)]} {s:.2f}" for c, s in zip(ref.boxes.cls, ref.boxes.conf)])
# каждой рамке Ultralytics найдётся наша того же класса с IoU > 0,9
ious = box_iou(ref.boxes.xyxy.cpu(), torch.tensor(mine, dtype=torch.float32))
same = ref.boxes.cls.cpu()[:, None] == classes[None, :]
assert ((ious * same).max(1).values > 0.9).all()

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img_rgb)
draw(ax, mine, [f"{names[int(c)]} {s:.2f}" for c, s in zip(classes, scores)], "tab:orange")
ax.set_title("YOLO11n, расшифровано своими руками: letterbox → сеть → NMS → обратно")
ax.axis("off")
plt.show()

Сколько рамок было бы без NMS? Посчитайте точки, прошедшие порог уверенности, — по нескольку на каждую кошку. YOLO26 убирает этот шаг: её o2o-голова учится выдавать одну рамку на объект (урок 17.5).

In [ ]:
print("точек с вероятностью > 0,25:", int((raw[4:].max(0).values > 0.25).sum()), "→ после NMS:", len(boxes))

## 3. Обучение YOLO на своих данных (урок 17.8)

Настоящий набор собирают и размечают неделями. Чтобы пройти весь путь за минуты, возьмём готовый маленький набор Ultralytics **medical-pills**: 92 картинки для обучения и 23 для валидации, один класс — таблетка (`pill`). Такого класса в COCO нет, так что YOLO26n придётся по-настоящему **дообучиться**. Ultralytics скачает набор сам (≈8 МБ).

Почему не coco8 (4 картинки)? При `batch=16` это один батч на эпоху, а Ultralytics делает шаг оптимизатора раз в 64 картинки (номинальный батч, урок 17.8): за несколько эпох не случится ни одного шага, и модель не изменится.

In [ ]:
from ultralytics.data.utils import check_det_dataset

info = check_det_dataset("medical-pills.yaml")
root = Path(info["path"])
for part in ("images/train", "labels/train", "images/val", "labels/val"):
    files = sorted(os.listdir(root / part))
    print(f"{part:13s} {len(files):3d} файлов:", files[:3])
label_file = sorted((root / "labels/val").glob("*.txt"))[0]
print("\n" + label_file.name + " (первые строки):")
print("\n".join(label_file.read_text().splitlines()[:4]))

Заметьте имена файлов: `Frame_0`, `Frame_100`… в обучении и `Frame_144`, `Frame_172`… в валидации — это кадры **одного видео**. Соседние кадры почти одинаковы, поэтому метрики на такой валидации, скорее всего, завышены (урок 17.8). Для учебного примера это допустимо, для своей задачи — нет.

**Задание 4.** Допишите `read_yolo_labels(path, W, H)`: прочитайте файл разметки YOLO (строки `класс cx cy w h`, доли от размера картинки) и верните список `(класс, x1, y1, x2, y2)` в пикселях. Проверка — рамки поверх картинки: они должны плотно сидеть на таблетках.

In [ ]:
def read_yolo_labels(path, W, H):
    out = []
    for line in Path(path).read_text().splitlines():
        if not line.strip():
            continue
        ...  # ваш код
    return out


img_file = root / "images/val" / (label_file.stem + ".jpg")
im = cv2.cvtColor(cv2.imread(str(img_file)), cv2.COLOR_BGR2RGB)
labels = read_yolo_labels(label_file, im.shape[1], im.shape[0])
assert len(labels) == sum(1 for line in label_file.read_text().splitlines() if line.strip())
assert all(0 <= x1 < x2 <= im.shape[1] + 1 and 0 <= y1 < y2 <= im.shape[0] + 1 for _, x1, y1, x2, y2 in labels)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.imshow(im)
draw(ax, [l[1:] for l in labels])
ax.set_title(f"{img_file.name}: {len(labels)} таблеток в разметке — проверяем глазами до обучения")
ax.axis("off")
plt.show()

Свой набор описывают файлом `data.yaml`. Сделаем такой файл сами — так же вы опишете свои картинки.

In [ ]:
cfg = {"path": str(root), "train": "images/train", "val": "images/val", "names": {0: "pill"}}
my_yaml = os.path.abspath("pills.yaml")
with open(my_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
print(open(my_yaml, encoding="utf-8").read())

Обучаем. На видеокарте — 30 эпох на 640 × 640 (несколько минут в Colab), на процессоре — 10 эпох на 320 × 320 (около минуты). Начинаем с предобученной YOLO26n: Ultralytics перенесёт все веса, кроме последних слоёв классов — их число изменилось с 80 на 1. Результаты лягут в папку `runs_m17/pills`.

In [ ]:
EPOCHS, IMGSZ = (30, 640) if device == "cuda" else (10, 320)
DEV = 0 if device == "cuda" else "cpu"
model = YOLO("yolo26n.pt")
t0 = time.time()
model.train(data=my_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=16, workers=2 if device == "cuda" else 0,
            device=DEV, project=os.path.abspath("runs_m17"), name="pills", exist_ok=True)
save_dir = Path(model.trainer.save_dir)
print(f"обучение: {time.time() - t0:.0f} с, папка: {save_dir}")
print(sorted(p.name for p in save_dir.iterdir()))

Таблица `results.csv`: потери на обучении и валидации и метрики по эпохам. У YOLO26 третья потеря — `l1_loss` (у YOLOv8 и YOLO11 на её месте `dfl_loss`).

In [ ]:
df = pd.read_csv(save_dir / "results.csv")
df.columns = df.columns.str.strip()
print(df[["epoch", "train/box_loss", "train/cls_loss", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].round(3).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for col in [c for c in df.columns if c.startswith(("train/", "val/"))]:
    axes[0].plot(df["epoch"], df[col], "-o", ms=3, label=col)
axes[0].set_title("потери")
axes[0].legend(fontsize=7)
for col in ("metrics/mAP50(B)", "metrics/mAP50-95(B)"):
    axes[1].plot(df["epoch"], df[col], "-o", ms=3, label=col)
axes[1].set_title("метрики на валидации")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xlabel("эпоха")
plt.show()
assert df["train/cls_loss"].iloc[-1] < df["train/cls_loss"].iloc[0]  # модель чему-то научилась

Проверим лучшие веса отдельно. Валидация по умолчанию берёт порог уверенности 0,001 — так mAP считается по всей кривой точность — полнота. Но матрицу ошибок Ultralytics строит при том же пороге, и в ней тысячи «ложных срабатываний» — рамок с уверенностью 0,01. Для разбора ошибок запустим валидацию ещё раз с рабочим порогом 0,25.

In [ ]:
best = YOLO(save_dir / "weights" / "best.pt")
common = dict(data=my_yaml, imgsz=IMGSZ, batch=16, workers=0, device=DEV, project=os.path.abspath("runs_m17"),
              exist_ok=True, verbose=False)
metrics = best.val(name="pills_val", **common)
print(f"mAP50 = {metrics.box.map50:.3f}, mAP50-95 = {metrics.box.map:.3f}")
assert metrics.box.map50 > 0.5

check = best.val(name="pills_conf25", conf=0.25, **common)
cm = check.confusion_matrix.matrix  # строки — предсказание, столбцы — истина; последние — background
print(f"найдено верно: {cm[0, 0]:.0f}, ложных срабатываний: {cm[0, 1]:.0f}, пропусков: {cm[1, 0]:.0f}")

plt.figure(figsize=(6, 5))
plt.imshow(plt.imread(Path(check.save_dir) / "confusion_matrix.png"))
plt.axis("off")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, name in zip(axes, ("val_batch0_labels.jpg", "val_batch0_pred.jpg")):
    ax.imshow(plt.imread(Path(check.save_dir) / name))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

Ложных срабатываний много. Разберём их глазами на той картинке, разметку которой мы рисовали в задании 4: сколько таблеток нашла модель и сколько их в разметке?

In [ ]:
res = best.predict(str(img_file), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
print(f"найдено таблеток: {len(res.boxes)}, в разметке: {len(labels)}")
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].imshow(im)
draw(axes[0], [l[1:] for l in labels])
axes[0].set_title("разметка")
axes[1].imshow(res.plot(labels=False)[..., ::-1])
axes[1].set_title("предсказания best.pt (conf ≥ 0,25)")
for ax in axes:
    ax.axis("off")
plt.show()

Сравните картинки. Многие «лишние» рамки модели стоят на **настоящих таблетках**, которых нет в разметке: у верхнего края, под блистером справа. Это не ошибки модели, а пропуски разметки — те самые «ошибки, где модель права» из урока 17.8. Есть и настоящие ложные срабатывания — например, на пустых ячейках блистера у края кадра. В своём проекте следующим шагом было бы исправить разметку и обучить заново.